# 02 — CMS i18n

The CMS message catalog: every locale complete, ICU placeholders
consistent, every message id used by the CMS present in the catalog,
and the locale resolution order (Zitadel preferredLanguage first).


In [ ]:
import jaen_testkit as k
k.start_run('02-cms-i18n')
print(k.CONFIG['repo_root'])

In [ ]:
import re

CATALOG = k.repo_path('packages/gatsby-plugin-jaen/src/locales/i18nJaen.ts')
src = k.read_text(CATALOG, '')

def split_blocks(source):
    """Return {locale: block_text}. The final return block is en-US."""
    blocks = {}
    positions = [(m.start(), m.group(1))
                 for m in re.finditer(r"if \(code === '([a-z]{2}-[A-Z]{2})'\)", source)]
    for i, (pos, locale) in enumerate(positions):
        end = positions[i + 1][0] if i + 1 < len(positions) else source.rfind('\n  return {')
        blocks[locale] = source[pos:end]
    blocks['en-US'] = source[source.rfind('\n  return {'):]
    return blocks

def keys_of(block):
    return re.findall(r"^\s+([A-Za-z][A-Za-z0-9]*):\s*$|^\s+([A-Za-z][A-Za-z0-9]*):\s*['\"]",
                      block, re.M)

def flat_keys(block):
    found = []
    for a, b in keys_of(block):
        key = a or b
        if key not in ('code', 'strings'):
            found.append(key)
    return found

blocks = split_blocks(src)
key_sets = {locale: set(flat_keys(block)) for locale, block in blocks.items()}

with k.section('catalog completeness'):
    with k.check('all configured CMS locales present') as c:
        for locale in k.CONFIG['cms_locales']:
            c.expect_true(locale in blocks, locale)

    baseline = key_sets.get('en-US', set())
    with k.check('en-US catalog is non-trivial') as c:
        c.expect_true(len(baseline) >= 150, '%d keys' % len(baseline))

    for locale, keys in sorted(key_sets.items()):
        if locale == 'en-US':
            continue
        with k.check('locale %s has the full key set' % locale) as c:
            missing = baseline - keys
            extra = keys - baseline
            c.expect_equal(sorted(missing), [], 'missing keys')
            c.expect_equal(sorted(extra), [], 'extra keys')


In [ ]:
def values_of(block):
    values = {}
    for m in re.finditer(
            r"^\s+([A-Za-z][A-Za-z0-9]*):\s*\n?\s*('(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\")",
            block, re.M):
        values[m.group(1)] = m.group(2)[1:-1]
    return values

en_values = values_of(blocks['en-US'])

with k.section('ICU placeholders'):
    placeholder = re.compile(r'\{(\w+)\}')
    for locale, block in sorted(blocks.items()):
        if locale == 'en-US':
            continue
        values = values_of(block)
        with k.check('placeholders consistent in %s' % locale) as c:
            broken = []
            for key, en_value in en_values.items():
                wanted = set(placeholder.findall(en_value))
                if not wanted:
                    continue
                got = set(placeholder.findall(values.get(key, '')))
                if wanted != got:
                    broken.append('%s: %s vs %s' % (key, sorted(wanted), sorted(got)))
            c.expect_equal(broken, [], 'all ICU placeholders preserved')
            if broken:
                c.detail('\n'.join(broken[:10]))


In [ ]:
import glob, os

with k.section('used ids exist'):
    used = set()
    for pattern in ('packages/gatsby-plugin-jaen/src/**/*.tsx',
                    'packages/gatsby-plugin-jaen/src/**/*.ts'):
        for path in glob.glob(k.repo_path(pattern), recursive=True):
            text = k.read_text(path, '')
            used.update(re.findall(r"(?:id:|intlText\()\s*'((?:Cms|Auth|Language)[A-Za-z0-9]*)'", text))
    with k.check('every used message id is in the catalog') as c:
        unknown = sorted(used - key_sets['en-US'])
        c.note('%d distinct ids used' % len(used))
        c.expect_equal(unknown, [], 'no unknown ids')
        if unknown:
            c.detail('\n'.join(unknown))


In [ ]:
with k.section('locale resolution'):
    wre = k.read_text(k.repo_path(
        'packages/gatsby-plugin-jaen/src/gatsby/wrap-root-element.tsx'), '')
    with k.check('zitadel preferredLanguage wins, then claim, then browser') as c:
        i1 = wre.find('matchLocale(preferredLanguage)')
        i2 = wre.find('matchLocale(claimLocale)')
        i3 = wre.find('matchLocale(browserLocale)')
        c.expect_true(0 <= i1 < i2 < i3, 'priority order in the resolver chain')
    with k.check('preferredLanguage sourced from the zitadel profile') as c:
        c.expect_contains(wre, 'human?.profile?.preferredLanguage')


In [ ]:
k.summary()
k.save_results('results-02-cms-i18n.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'